# Overview

This notebook demonstrates how to use Amazon Bedrock's data automation, agents, and knowledge bases to automate the insurance claim lifecycle. We'll show how to use Bedrock data automation to create a custom blueprint and process medical claims form (CMS 1500). We also demonstrate the end-to-end-flow of processing the claims data using a Bedrock Knowledge Base and Agents.

# Context

Insurance companies deal with a high volume of claims, which can be time-consuming and prone to errors when processed manually. By leveraging capabilities provided by Amazon Bedrock including Bedrock Data Automation and Bedrock Agents, We can create an AI-powered system that streamlines the claim process, improves efficiency, and enhances customer experience.

## Architecture
![Claims Review Architecture](data/images/Medical_Claims_Processing_Architecture.png)

# Prerequisites
Before starting this notebook, ensure you have:

1. An AWS account with access to Amazon Bedrock
2. Necessary IAM permissions to create and manage Bedrock resources
3. AWS SDK for Python (Boto3) installed
4. A sample dataset of insurance claims (we'll use a synthetic dataset for this demo)

# Setup
In the following sections we would run through the process to setup the AWS resources required to run the end-to-end flow for claim processing

In [1]:
%pip install "boto3>=1.35.76" itables==2.2.4 PyPDF2==3.0.1 --upgrade -q

Note: you may need to restart the kernel to use updated packages.


## Import Remote Libraries
First, let's import the necessary libraries.

We'll use the `us-west-2` region and create the boto3 clients for the required AWS services

In [2]:
%load_ext autoreload
%autoreload 1

In [3]:
import boto3
import json
import os
from datetime import datetime
import time
from botocore.exceptions import ClientError

%aimport utils.bedrock_utils
%aimport utils.helper_functions
from utils import helper_functions
from utils import bedrock_utils

from pathlib import Path
import os

from IPython.display import JSON, display, IFrame, HTML, Markdown

REGION_NAME = 'us-west-2'

# Set up AWS credentials (make sure you have the appropriate permissions)
session = boto3.Session(region_name=REGION_NAME)  # Replace with your preferred region
sts = session.client('sts')
iam = boto3.client('iam')
s3_client = session.client('s3')
bedrock = session.client('bedrock')
bedrock_agent = session.client('bedrock-agent')
bedrock_agent_runtime = session.client('bedrock-agent-runtime')
bda_client = boto3.client('bedrock-data-automation')
bda_runtime_client = boto3.client('bedrock-data-automation-runtime')


aoss_client = boto3.client('opensearchserverless')
account_id = sts.get_caller_identity()['Account']

### Fetch resource details from workshop infrastructure stack

## Create S3 Buckets

### S3 Bucket to serve as input for evidence of coverage documents

In [4]:
claims_eoc_bucket_name = f'claims-eoc-{account_id}'
try:
    s3_client.create_bucket(Bucket=claims_eoc_bucket_name, CreateBucketConfiguration={'LocationConstraint': REGION_NAME})
    print(f"Bucket '{claims_eoc_bucket_name}' created.")

except s3_client.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket '{claims_eoc_bucket_name}' already exists. Will use it")


Bucket 'claims-eoc-604593561756' already exists. Will use it


### S3 Bucket to serve as input bucket for claim forms

In [5]:
claims_submission_bucket_name = f'claims-submission-{account_id}'
try:
    s3_client.create_bucket(Bucket=claims_submission_bucket_name, CreateBucketConfiguration={'LocationConstraint': REGION_NAME})
    print(f"Bucket '{claims_submission_bucket_name}' created.")

except s3_client.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket '{claims_submission_bucket_name}' already exists. Will use it")


Bucket 'claims-submission-604593561756' already exists. Will use it


### Setup BDA Input / Output Locations in S3

In [6]:
bda_s3_input_location = f's3://{claims_submission_bucket_name}/input'
bda_s3_output_location = f's3://{claims_submission_bucket_name}/output'
agent_review_s3_input_location = f's3://{claims_submission_bucket_name}/agent_input'

### Setup Parameters for Knowledge Base and Data Source

In [7]:
embedding_model_arn = 'arn:aws:bedrock:us-west-2::foundation-model/amazon.titan-embed-text-v2:0'
foundation_model_id = 'anthropic.claude-3-5-haiku-20241022-v1:0'
kb_role_arn = helper_functions.get_stack_output(stack_name = 'bda-idp-workshop',output_key = 'KBServiceRole')
vector_store_index_name = helper_functions.get_stack_output(stack_name = 'bda-idp-workshop',output_key = 'ClaimsVectorStoreIndexName')
vector_store_collection_arn= helper_functions.get_stack_output(stack_name = 'bda-idp-workshop',output_key = 'ClaimsVectorStoreCollectionArn')
vector_store_collection_name = helper_functions.get_stack_output(stack_name = 'bda-idp-workshop',output_key = 'ClaimsVectorStoreCollectionName')
agent_service_role_arn = helper_functions.get_stack_output(stack_name = 'bda-idp-workshop',output_key = 'AgentServiceRole')
agent_actions_lambda_arn = helper_functions.get_stack_output(stack_name = 'bda-idp-workshop',output_key = 'ClaimsReviewAgentActionLambdaFunction')

### Create knowledge base for Evidence of Coverage documents

In [8]:
knowledge_base_id = bedrock_utils.create_knowledge_base(
    bedrock_agent,
    kb_name='claims-eoc-kb',
    kb_description='claims eoc kb',
    embedding_model_arn=embedding_model_arn,
    kb_role_arn=kb_role_arn,
    vector_store_collection_arn=vector_store_collection_arn,
    vector_store_index_name=vector_store_index_name)

Creating new KB with name claims-eoc-kb


### Create data source to ingest evidence of coverage documents

In [9]:
data_source_id = bedrock_utils.create_data_source(bedrock_agent, knowledge_base_id, datasource_name='claims-eoc-datasource')

In [10]:
print(knowledge_base_id)
print(data_source_id)

DHIETNZ6PO
EEJDVUN4JL


### Upload EoC Documents to S3

In [11]:
!aws s3 cp data/documents/eoc/ s3://{claims_eoc_bucket_name} --recursive

upload: data/documents/eoc/Evidence_of_Coverage_-_AnyHealth_Standard.pdf to s3://claims-eoc-604593561756/Evidence_of_Coverage_-_AnyHealth_Standard.pdf
upload: data/documents/eoc/Evidence_of_Coverage_-_AnyHealth_Plus.pdf to s3://claims-eoc-604593561756/Evidence_of_Coverage_-_AnyHealth_Plus.pdf
upload: data/documents/eoc/Evidence_of_Coverage_-_AnyHealth_Premium.pdf to s3://claims-eoc-604593561756/Evidence_of_Coverage_-_AnyHealth_Premium.pdf


### Ingest EoC Documents Directly to our KB Datasource

#### Setup document configuration

In [18]:
document_1 = {
    'plan_name':'AnyHealth_Standard',
    'document_id': 'Evidence_of_Coverage_-_AnyHealth_Standard.pdf',
    'document_uri': f's3://{claims_eoc_bucket_name}/Evidence_of_Coverage_-_AnyHealth_Standard.pdf'
}
document_2 = {
    'plan_name':'AnyHealth_Premium',
    'document_id': 'Evidence_of_Coverage_-_AnyHealth_Premium.pdf',
    'document_uri': f's3://{claims_eoc_bucket_name}/Evidence_of_Coverage_-_AnyHealth_Premium.pdf'
}
document_3 = {
    'plan_name':'AnyHealth_Plus',
    'document_id': 'Evidence_of_Coverage_-_AnyHealth_Plus.pdf',
    'document_uri': f's3://{claims_eoc_bucket_name}/Evidence_of_Coverage_-_AnyHealth_Plus.pdf'
}


In [13]:
def ingest_and_wait(data_source_id , knowledge_base_id, document):
    ingest_response = bedrock_agent.ingest_knowledge_base_documents(
        dataSourceId=data_source_id,
        knowledgeBaseId=knowledge_base_id,
        documents=[
            bedrock_utils.get_document_configuration(document['document_id'], document['plan_name'], document['document_uri'])
        ]
    )
    status_response = helper_functions.wait_for_completion(
                client=bedrock_agent,
                get_status_function=bedrock_agent.get_knowledge_base_documents,
                status_kwargs={
                    'dataSourceId':data_source_id,
                    'knowledgeBaseId': knowledge_base_id,
                    'documentIdentifiers': [{
                        'custom': {
                            'id': document['document_id']
                        },
                        'dataSourceType': 'CUSTOM'}]
                },
                completion_states=['INDEXED'],
                error_states=['FAILED'],
                status_path_in_response='documentDetails[0].status',
                max_iterations=5,
                delay=15
    )

In [15]:
ingest_and_wait(data_source_id, knowledge_base_id, document_1)

Current status: STARTING. Waiting...
Operation completed successfully with status: INDEXED


In [16]:
ingest_and_wait(data_source_id, knowledge_base_id, document_2)

Current status: STARTING. Waiting...
Operation completed successfully with status: INDEXED


In [19]:
ingest_and_wait(data_source_id, knowledge_base_id, document_3)

Current status: STARTING. Waiting...
Operation completed successfully with status: INDEXED


## Read agent instruction

In [20]:
with open('data/agent_resources/agent_prompt.txt','r') as f:
    agent_instruction = f.read()

In [21]:
create_agent_response = bedrock_agent.create_agent(
    agentName='claims-agent',
    agentResourceRoleArn=agent_service_role_arn,
    description='claims review agent',
    foundationModel=foundation_model_id,
    instruction=agent_instruction,
    orchestrationType='DEFAULT'
)
agent_id = create_agent_response['agent']['agentId']

ConflictException: An error occurred (ConflictException) when calling the CreateAgent operation: Could not perform Create operation, since the claims-agent (id: YOTVQJFBPZ) with the same name claims-agent already exists

In [22]:
status_response = helper_functions.wait_for_completion(
                client=bedrock_agent,
                get_status_function=bedrock_agent.get_agent,
                status_kwargs={
                    'agentId': agent_id
                },
                completion_states=['NOT_PREPARED'],
                error_states=['FAILED'],
                status_path_in_response='agent.agentStatus',
                max_iterations=5,
                delay=15
    )

NameError: name 'agent_id' is not defined

In [23]:
bedrock_agent.prepare_agent(agentId=agent_id)
status_response = helper_functions.wait_for_completion(
                client=bedrock_agent,
                get_status_function=bedrock_agent.get_agent,
                status_kwargs={
                    'agentId': agent_id
                },
                completion_states=['PREPARED'],
                error_states=['FAILED'],
                status_path_in_response='agent.agentStatus',
                max_iterations=5,
                delay=15
    )

NameError: name 'agent_id' is not defined

In [24]:
with open('data/agent_resources/agent_action_schema.json') as f:
    api_schema = f.read()

In [25]:
create_action_group_response = bedrock_agent.create_agent_action_group(
    actionGroupExecutor={
        'lambda': agent_actions_lambda_arn
    },
    actionGroupName='claims-review-actions',
    actionGroupState='ENABLED',
    agentId=agent_id,
    agentVersion='DRAFT',
    apiSchema={
        'payload': api_schema,
    },
    description='claims review agent actions'
)

NameError: name 'agent_id' is not defined

In [ ]:
associate_agent_kb_response = bedrock_agent.associate_agent_knowledge_base(
    agentId=agent_id,
    agentVersion='DRAFT',
    description='Claims Evidence of Coverage, used to verify if claims are covered under specific coverage plan terms',
    knowledgeBaseId=knowledge_base_id,
    knowledgeBaseState='ENABLED'
)
associate_agent_kb_response

In [ ]:
create_agent_alias_response = bedrock_agent.create_agent_alias(
    agentAliasName='LIVE',
    agentId=agent_id,
    description='LIVE version of agent'
)

In [ ]:
agent_alias_id = create_agent_alias_response['agentAlias']['agentAliasId']

## Prepare sample claims form document

For this lab, we use a sample `Medical Claim` form.

In [ ]:
local_download_path = 'data/documents'
local_file_name = 'sample1_cms-1500-P.pdf'
local_file_path = os.path.join(local_download_path, local_file_name)

document_s3_uri = f'{bda_s3_input_location}/{local_file_name}'

target_s3_bucket, target_s3_key =  helper_functions.get_bucket_and_key(document_s3_uri)
s3_client.upload_file(local_file_path, target_s3_bucket, target_s3_key)

print(f"Downloaded file to: {local_file_path}")
print(f"Uploaded file to S3: {target_s3_key}")
print(f"document_s3_uri: {document_s3_uri}")

### View Sample Document

In [ ]:
IFrame(local_file_path, width=800, height=600)

### Create Blueprint for Medical Claim form (CMS 1500)

In [ ]:
blueprint = {
    "name": 'claim-form',
    "description": 'Blueprint for Medical Claim form CMS 1500',
    "type": 'DOCUMENT',
    "stage": 'LIVE',
    "schema_path": 'data/blueprint/claims_form.json'
}
with open(blueprint['schema_path']) as f:
    blueprint_schema = json.load(f)
    blueprint_arn = helper_functions.create_or_update_blueprint(
        bda_client, 
        blueprint['name'], 
        blueprint['description'], 
        blueprint['type'],
        blueprint['stage'],
        blueprint_schema
    )
blueprint_arn

In [ ]:
response = bda_runtime_client.invoke_data_automation_async(
    inputConfiguration={
        's3Uri': document_s3_uri
    },
    outputConfiguration={
        's3Uri': bda_s3_output_location
    },
    blueprints=[
        {
            'blueprintArn': blueprint_arn
        }
    ]
)

invocationArn = response['invocationArn']
print(f'Invoked data automation job with invocation arn {invocationArn}')

In [ ]:
status_response = helper_functions.wait_for_completion(
            client=bda_client,
            get_status_function=bda_runtime_client.get_data_automation_status,
            status_kwargs={'invocationArn': invocationArn},
            completion_states=['Success'],
            error_states=['ClientError', 'ServiceError'],
            status_path_in_response='status',
            max_iterations=15,
            delay=30
)
if status_response['status'] == 'Success':
    job_metadata_s3_location = status_response['outputConfiguration']['s3Uri']
else:
    raise Exception(f'Invocation Job Error, error_type={status_response["error_type"]},error_message={status_response["error_message"]}')

In [ ]:
job_metadata = json.loads(helper_functions.read_s3_object(job_metadata_s3_location))
job_id = job_metadata['job_id']
JSON(job_metadata, root='job_metadata', expanded=True)

In [ ]:
asset_id = 0
custom_output_path = next(item["segment_metadata"][0]["custom_output_path"] 
                                for item in job_metadata["output_metadata"] 
                                if item['asset_id'] == asset_id)
custom_output = json.loads(helper_functions.read_s3_object(custom_output_path))

In [ ]:
s3_client.put_object(
    Bucket=claims_submission_bucket_name,
    Key = f'agent_input/{job_id}.json',
    Body=json.dumps(custom_output)
)

### Invoke Agent for Claim Verification

In [ ]:
import uuid
response = bedrock_agent_runtime.invoke_agent(
    agentAliasId=agent_alias_id,
    agentId=agent_id,
    enableTrace=True,
    inputText=f"Review the claim using claim form data in S3 URI s3://{claims_submission_bucket_name}/agent_input/{job_id}",
    memoryId='string',
    sessionId=str(uuid.uuid4())
)

In [ ]:
for event in response.get("completion"):
    if "chunk" in event.keys():
        chunk = event["chunk"]
        completion = completion + chunk["bytes"].decode()
    elif("trace" in event.keys()):
        print(event["trace"]["trace"])

completion

In [ ]:
Markdown(completion)

## Next Steps